# Notebook 12e — Hetero vs Uniform VLA: Clean Capacity Experiment

## What's different from 12d and why
| Issue in 12d | Fix here |
|---|---|
| DeltaNet engine | VLA Sherman–Morrison engine (what we’re actually claiming) |
| T=max(4n,512) inflated sequence | Standard T=3n+1 (the only format proven to converge) |
| VOCAB=1024, values in [512,1022] | VOCAB=128 (every prior success used this) |
| Batched (H,dh,dh) init bug | Per-head nn.Linear — correct fan_in guaranteed |
| GRAD_CLIP=1.0 too tight at scale | GRAD_CLIP=5.0 (measured norms avg 3–8 in our tests) |
| No gradient checkpointing → OOM at n≥32 | Chunked grad checkpoint + dynamic batch + AMP |

## OOM fixes (v2)
- **Gradient checkpointing**: VLA recurrence chunked every 32 steps; only one chunk’s intermediates live at a time
- **Dynamic batch**: 64→32→16→8 as n grows, with gradient accumulation to preserve effective batch=64
- **AMP**: float16 for linear projections & FFN; float32 for the numerically-sensitive Sherman–Morrison recurrence
- **Separate cells**: each n_pairs value runs in its own cell for independent execution & logging

## The claim being tested
Hetero-VLA (4×d_h=32 fast + 4×d_h=256 slow, capacity=1152) maintains >0.8
accuracy at n_pairs=200 while Uniform-VLA (8×d_h=128, capacity=1024)
collapses well before that.

## Why these choices
VOCAB=128 works because we proved VLA reaches 1.000 at n=8 with these
settings in notebook 09. Standard MQAR T=3n+1 works because every single
successful training run in this project used it.

## 0 · Setup

In [ ]:
import math, time, gc, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as ckpt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT    = Path('/kaggle/working/nb12e')
for s in ['plots','logs']: (OUT/s).mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family':'DejaVu Serif','font.size':11,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.25,'figure.dpi':150,
})
C = {'uniform':'#55EFC4','hetero':'#00B894','deltanet':'#FDCB6E'}

print(f'Device: {DEVICE}  Torch: {torch.__version__}')

## 1 · Config

In [ ]:
# ── Proven working settings ────────────────────────────────────────────────
VOCAB     = 128    # CRITICAL: every successful run used this. Random baseline=1/127
BATCH     = 64     # effective batch (reached via gradient accumulation)
LR        = 1e-3   # matches nb09 that reached 1.000
WARMUP    = 150
GRAD_CLIP = 5.0    # raised from 1.0: measured pre-clip norms of 3-8 at this scale
CHUNK     = 32     # gradient-checkpoint chunk size for VLA recurrence

# ── Architecture: the roadmap design ──────────────────────────────────────
FAST_H, FAST_DH = 4, 32    # fast heads
SLOW_H, SLOW_DH = 4, 256   # slow heads
D_HETERO = FAST_H*FAST_DH + SLOW_H*SLOW_DH  # 128 + 1024 = 1152

UNIF_H, UNIF_DH = 8, 128
D_UNIF = UNIF_H * UNIF_DH   # 1024

# ── Sweep ──────────────────────────────────────────────────────────────────
N_PAIRS = [8, 32, 128, 200]
# 8   -> sanity: VLA must reach >0.9 or architecture is still broken
# 32  -> within capacity for both (confirmed working in nb09)
# 128 -> Uniform-VLA boundary; expect it to start struggling
# 200 -> THE claim: Hetero >0.80, Uniform <<0.80
SEEDS = [42, 123, 999]

def steps_for_n(n):
    if n <= 8:   return 1500
    if n <= 32:  return 2000
    if n <= 128: return 2800
    return 3500

def batch_for_n(n):
    """Dynamic batch size to keep GPU memory bounded on T4 (15 GB).
    Gradient accumulation restores the effective batch to BATCH."""
    if n <= 8:   return 64
    if n <= 32:  return 32
    if n <= 128: return 16
    return 8

N_LAYERS = 2

print(f'VOCAB={VOCAB}  BATCH={BATCH}  LR={LR}  GRAD_CLIP={GRAD_CLIP}  CHUNK={CHUNK}')
print(f'Hetero cap={FAST_H*FAST_DH + SLOW_H*SLOW_DH}  Uniform cap={UNIF_H*UNIF_DH}')
print(f'Random baseline = {1/(VOCAB-1):.4f}')
for n in N_PAIRS:
    T = 3*n+1
    print(f'  n={n:4d}: T={T:4d}  steps={steps_for_n(n):,}  batch={batch_for_n(n)}')

## 2 · Models (per-head nn.Linear + gradient checkpointing)

In [ ]:
# ── VLAv3 single-head with CHUNKED GRADIENT CHECKPOINTING ────────────
# Key change from v1: the T-step recurrence is split into CHUNK-sized
# segments.  Each segment is wrapped in torch.utils.checkpoint so that
# only one chunk's intermediates are live at a time during backward.
# This reduces activation memory from O(T * dh^2) to O(CHUNK * dh^2).

class VLAv3Head(nn.Module):
    def __init__(self, dh, lam=0.1, eps=1e-4, per_eps=1e-3, period=20, chunk=CHUNK):
        super().__init__()
        self.dh=dh; self.lam=lam; self.eps=eps
        self.per_eps=per_eps; self.period=period; self.chunk=chunk
        # Each a standard nn.Linear(dh,dh) — PyTorch uses kaiming_uniform
        # with fan_in=dh, which is CORRECT. No batched-param bug here.
        self.Wq=nn.Linear(dh,dh); self.Wk=nn.Linear(dh,dh)
        self.Wv=nn.Linear(dh,dh); self.Wu=nn.Linear(dh,dh,bias=False)
        self.Wo=nn.Linear(dh,dh); self.norm=nn.LayerNorm(dh)

    def _recurrence_chunk(self, kf_c, Q_c, V_c, U_c, A, S, zk, I, c_start):
        """Process one chunk of the Sherman–Morrison recurrence in float32.
        Called via ckpt.checkpoint — intermediates are freed after each chunk."""
        # Disable autocast so bmm/einsum stay in float32 for numerical safety
        with torch.amp.autocast(device_type=kf_c.device.type, enabled=False):
            B, C_len, d = kf_c.shape
            isq = 1/math.sqrt(d)
            ys = []
            for t in range(C_len):
                gt = c_start + t
                u = U_c[:,t,:] * isq
                zsm = torch.bmm(A, u.unsqueeze(-1)).squeeze(-1)
                dlt = (1 + (u*zsm).sum(-1)).clamp(min=self.eps)
                A = A - torch.einsum('bi,bj->bij', zsm, zsm) / dlt.view(B,1,1)
                if (gt+1) % self.period == 0:
                    A = A + self.per_eps * I.unsqueeze(0)
                kn = F.normalize(kf_c[:,t,:], p=2, dim=-1)
                alpha = torch.bmm(A, kn.unsqueeze(-1)).squeeze(-1)
                alphn = F.normalize(alpha, p=2, dim=-1)
                e = V_c[:,t,:] - torch.bmm(S, kn.unsqueeze(-1)).squeeze(-1)
                S = S + torch.einsum('bi,bj->bij', e, alphn)
                qt = Q_c[:,t,:]
                zk = zk + kf_c[:,t,:]
                yt = torch.bmm(S, qt.unsqueeze(-1)).squeeze(-1)
                ys.append(yt / (zk*qt).sum(-1, keepdim=True).clamp(min=self.eps))
            return torch.stack(ys, 1), A, S, zk

    def forward(self, x):
        B, T, d = x.shape
        # ── Projections (benefit from AMP float16 if active) ──
        kr = self.Wk(x); kf = F.elu(kr) + 1.0
        Q  = F.elu(self.Wq(x)) + 1.0; V = self.Wv(x)
        U  = F.normalize(self.Wu(kr), p=2, dim=-1)
        # ── Cast to float32 for numerically stable recurrence ──
        kf, Q, V, U = kf.float(), Q.float(), V.float(), U.float()
        I  = torch.eye(d, device=x.device, dtype=torch.float32)
        A  = (1/self.lam) * I.unsqueeze(0).expand(B,-1,-1).clone()
        S  = torch.zeros(B, d, d, device=x.device, dtype=torch.float32)
        zk = torch.zeros(B, d, device=x.device, dtype=torch.float32)
        # ── Chunked recurrence with gradient checkpointing ──
        all_ys = []
        for c0 in range(0, T, self.chunk):
            c1 = min(c0 + self.chunk, T)
            chunk_ys, A, S, zk = ckpt.checkpoint(
                self._recurrence_chunk,
                kf[:,c0:c1], Q[:,c0:c1], V[:,c0:c1], U[:,c0:c1],
                A, S, zk, I, c0,
                use_reentrant=False
            )
            all_ys.append(chunk_ys)
        return self.Wo(self.norm(torch.cat(all_ys, 1)))


# ── Multi-head wrappers ────────────────────────────────────────────────────
class UniformVLA(nn.Module):
    """Standard uniform VLA: H heads, all dh = d_model // H."""
    def __init__(self, d_model, H=UNIF_H):
        super().__init__()
        self.H=H; self.dh=d_model//H
        self.heads=nn.ModuleList([VLAv3Head(self.dh) for _ in range(H)])
        self.Wo=nn.Linear(d_model,d_model); self.norm=nn.LayerNorm(d_model)
    def capacity(self): return self.H*self.dh
    def forward(self,x):
        B,T,D=x.shape; dh=self.dh; outs=[]
        for i,h in enumerate(self.heads):
            outs.append(h(x[:,:,i*dh:(i+1)*dh]))
        return self.Wo(self.norm(torch.cat(outs,-1)))


class HeteroVLA(nn.Module):
    """Heterogeneous VLA: fast heads (small dh) + slow heads (large dh).
    Capacity = fast_H*fast_dh + slow_H*slow_dh.
    Prop 2 holds per head independently (each head is a VLAv3Head).
    Corollary: total capacity = sum of per-head capacities.
    """
    def __init__(self,
                 fast_H=FAST_H, fast_dh=FAST_DH,
                 slow_H=SLOW_H, slow_dh=SLOW_DH):
        super().__init__()
        self.fast_H=fast_H; self.fast_dh=fast_dh
        self.slow_H=slow_H; self.slow_dh=slow_dh
        self._fast_dim = fast_H*fast_dh
        self._d = fast_H*fast_dh + slow_H*slow_dh
        self.fast_heads=nn.ModuleList([VLAv3Head(fast_dh) for _ in range(fast_H)])
        self.slow_heads=nn.ModuleList([VLAv3Head(slow_dh) for _ in range(slow_H)])
        self.Wo=nn.Linear(self._d,self._d); self.norm=nn.LayerNorm(self._d)
    def capacity(self): return self.fast_H*self.fast_dh + self.slow_H*self.slow_dh
    def forward(self,x):
        fd=self._fast_dim; outs=[]
        xf=x[:,:,:fd]; xs=x[:,:,fd:]
        for i,h in enumerate(self.fast_heads):
            outs.append(h(xf[:,:,i*self.fast_dh:(i+1)*self.fast_dh]))
        for i,h in enumerate(self.slow_heads):
            outs.append(h(xs[:,:,i*self.slow_dh:(i+1)*self.slow_dh]))
        return self.Wo(self.norm(torch.cat(outs,-1)))


# ── Backbone ──────────────────────────────────────────────────────────────
class Block(nn.Module):
    def __init__(self,attn,d,ff=2):
        super().__init__()
        self.ln1=nn.LayerNorm(d); self.ln2=nn.LayerNorm(d)
        self.attn=attn
        self.ff=nn.Sequential(nn.Linear(d,d*ff),nn.GELU(),nn.Linear(d*ff,d))
    def forward(self,x): return x+self.ff(self.ln2(x+self.attn(self.ln1(x))))

class TinyLM(nn.Module):
    def __init__(self,attn_fn,d,vocab=VOCAB,n_layers=N_LAYERS):
        super().__init__()
        self.tok=nn.Embedding(vocab,d); self.pos=nn.Embedding(8192,d)
        self.blocks=nn.ModuleList([Block(attn_fn(d),d) for _ in range(n_layers)])
        self.ln_f=nn.LayerNorm(d)
        self.head=nn.Linear(d,vocab,bias=False); self.head.weight=self.tok.weight
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.normal_(m.weight,std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m,nn.Embedding): nn.init.normal_(m.weight,std=0.02)
    def forward(self,idx):
        B,T=idx.shape
        x=self.tok(idx)+self.pos(torch.arange(T,device=idx.device).unsqueeze(0))
        for b in self.blocks: x=b(x)
        return self.ln_f(x)@self.tok.weight.T


MODEL_REGISTRY = {
    'Uniform-VLA': (lambda d: UniformVLA(d,H=UNIF_H), D_UNIF,  C['uniform']),
    'Hetero-VLA':  (lambda d: HeteroVLA(),             D_HETERO, C['hetero']),
}

print('NaN + checkpoint sanity:')
for name,(factory,d_model,_) in MODEL_REGISTRY.items():
    m=TinyLM(factory,d_model,VOCAB).to(DEVICE)
    x=torch.randint(0,VOCAB,(2,25),device=DEVICE)
    with torch.amp.autocast('cuda', enabled=(DEVICE=='cuda')):
        o=m(x)
    p=sum(q.numel() for q in m.parameters())/1e6
    cap=factory(d_model).capacity() if hasattr(factory(d_model),'capacity') else '?'
    print(f'  {name:14s}: NaN={torch.isnan(o).any().item()}  params={p:.2f}M  '
          f'd={d_model}  capacity={cap}')
    del m; gc.collect()
    if DEVICE=='cuda': torch.cuda.empty_cache()

## 3 · MQAR Task (standard format, T=3n+1)

In [ ]:
def make_mqar(B, n_pairs, vocab=VOCAB, device=DEVICE):
    """Standard MQAR: [k1 v1 ... kn vn SEP q1 ... qn].  T = 3n+1.
    This is the ONLY format that has ever converged in this project.
    """
    sep       = vocab-1
    key_range = max(vocab-1, n_pairs+1)
    T         = 2*n_pairs+1+n_pairs
    x=torch.full((B,T),sep,dtype=torch.long,device=device)
    y=torch.full((B,T),-100,dtype=torch.long,device=device)
    for b in range(B):
        raw=torch.randperm(key_range,device=device)[:n_pairs]
        keys=raw%(vocab-1)
        vals=torch.randint(0,vocab-1,(n_pairs,),device=device)
        for i in range(n_pairs):
            x[b,2*i]=keys[i]; x[b,2*i+1]=vals[i]
        x[b,2*n_pairs]=sep
        perm=torch.randperm(n_pairs,device=device)
        x[b,2*n_pairs+1:]=keys[perm]
        y[b,2*n_pairs+1:]=vals[perm]
    return x,y

print('MQAR format checks:')
for n in [8,32,128,200]:
    x,y=make_mqar(4,n)
    T_exp=3*n+1
    assert x.shape==(4,T_exp)
    assert (y!=-100).sum().item()==4*n
    assert x.max().item()<=VOCAB-1
    print(f'  n={n:4d}: T={T_exp:4d}  targets={4*n}  OK')
print(f'Random baseline = {1/(VOCAB-1):.4f}')

## 4 · Training Loop (AMP + dynamic batch + grad accum) & Sanity Check

In [ ]:
# This cell defines the core training function and runs a sanity check.
# If Uniform-VLA can't reach >0.85 at n=8, the architecture has a bug
# and we stop here rather than wasting GPU on the sweep.

def run_mqar(attn_fn, d_model, n_pairs, steps,
             batch=None, lr=LR, seed=42,
             log_every=200, verbose=True):
    """Train & evaluate one (model, n_pairs) setting.

    OOM-safe: uses dynamic batch size (batch_for_n), gradient accumulation
    to maintain effective batch=BATCH, AMP for projections/FFN, and gradient
    checkpointing inside VLAv3Head.
    """
    if batch is None:
        batch = batch_for_n(n_pairs)
    accum = max(1, BATCH // batch)
    use_amp = (DEVICE == 'cuda')

    torch.manual_seed(seed)
    model = TinyLM(attn_fn, d_model, VOCAB).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scaler = torch.amp.GradScaler(enabled=use_amp)

    def lrf(s):
        if s < WARMUP: return s / max(WARMUP, 1)
        return 0.5 * (1 + math.cos(math.pi * (s - WARMUP) / max(steps - WARMUP, 1)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lrf)

    rand_base = 1 / (VOCAB - 1); streak = 0; t0 = time.time()
    model.train()

    for step in range(1, steps + 1):
        opt.zero_grad(set_to_none=True)
        step_loss = 0.0
        nan_hit = False

        for _ in range(accum):
            x, y = make_mqar(batch, n_pairs)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, VOCAB),
                                       y.view(-1), ignore_index=-100) / accum
            if not torch.isfinite(loss):
                nan_hit = True
                break
            scaler.scale(loss).backward()
            step_loss += loss.item()

        if nan_hit:
            if verbose:
                print(f'      NaN at step {step}')
            break

        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(opt); scaler.update()
        sched.step()

        if step % log_every == 0 or step == 1:
            model.eval()
            with torch.no_grad(), torch.amp.autocast('cuda', enabled=use_amp):
                pa = []
                for _ in range(3):
                    xp, yp = make_mqar(batch, n_pairs)
                    mp = yp != -100
                    if mp.any():
                        pa.append((model(xp).argmax(-1)[mp] == yp[mp]).float().mean().item())
                probe = float(np.mean(pa)) if pa else 0.0
            model.train()
            flag = '' if probe > rand_base * 3 else '  <- still ~random'
            if verbose:
                print(f'      step {step:5d}/{steps}  loss={step_loss*accum:.4f}  '
                      f'acc={probe:.4f}  ({time.time()-t0:.0f}s){flag}')
            if probe >= 0.95:
                streak += 1
            else:
                streak = 0
            if streak >= 3:
                if verbose:
                    print(f'      [converged early at step {step}]')
                break

    # ── Final evaluation (20 batches) ──
    model.eval()
    accs = []
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=use_amp):
        for _ in range(20):
            xv, yv = make_mqar(batch, n_pairs)
            mask = yv != -100
            if mask.any():
                accs.append((model(xv).argmax(-1)[mask] == yv[mask]).float().mean().item())
    result = float(np.mean(accs)) if accs else 0.0

    del model, opt, scaler
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return result


# ── Sanity check ─────────────────────────────────────────────────────
print('SANITY CHECK: Uniform-VLA on n=8  (expected >0.85 by step ~800)')
print('=' * 60)
acc_sanity = run_mqar(
    lambda d: UniformVLA(d, H=UNIF_H), D_UNIF,
    n_pairs=8, steps=1500, log_every=100, seed=42)
print()
print(f'Sanity result: {acc_sanity:.4f}')
if acc_sanity > 0.85:
    print('PASS -- architecture is healthy. Run the sweep.')
else:
    print('FAIL -- DO NOT run the sweep. Architecture still has a bug.')
    print('       Check: is VOCAB=128? Is T=3n+1? Is per-head init used?')

## 5a · Sweep: n=8

In [ ]:
# ── Sweep n=8 ───────────────────────────────────────────────────────────
n = 8
st = steps_for_n(n)
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
print()
print('=' * 60)
print(f' n={n}  steps={st:,}  T={3*n+1}  batch={batch_for_n(n)}  accum={BATCH//batch_for_n(n)}')
print('=' * 60)

_fp = OUT / 'logs' / f'n{n}.csv'
_recovered = pd.read_csv(_fp).to_dict('records') if _fp.exists() else []
if _recovered:
    print(f'  Recovered {len(_recovered)} rows from {_fp.name}')

_rows = list(_recovered)
for mname, (factory, d_model, _) in MODEL_REGISTRY.items():
    if any(r['model'] == mname for r in _recovered):
        print(f'  Skipping {mname} (already done)')
        continue
    accs = []
    print(f'  [{mname}]')
    for seed in SEEDS:
        acc = run_mqar(factory, d_model, n, st, seed=seed,
                       log_every=max(300, st // 8))
        accs.append(round(acc, 4))
        print(f'    seed={seed}: {acc:.4f}')
    mean, std = float(np.mean(accs)), float(np.std(accs))
    print(f'    MEAN={mean:.4f} +/- {std:.4f}')
    _rows.append({'n_pairs': n, 'model': mname,
                  'mean': round(mean, 4), 'std': round(std, 4)})
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

pd.DataFrame(_rows).to_csv(_fp, index=False)
print(f'  Checkpoint: n{n}.csv')
if DEVICE == 'cuda':
    print(f'  Peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## 5b · Sweep: n=32

In [ ]:
# ── Sweep n=32 ───────────────────────────────────────────────────────────
n = 32
st = steps_for_n(n)
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
print()
print('=' * 60)
print(f' n={n}  steps={st:,}  T={3*n+1}  batch={batch_for_n(n)}  accum={BATCH//batch_for_n(n)}')
print('=' * 60)

_fp = OUT / 'logs' / f'n{n}.csv'
_recovered = pd.read_csv(_fp).to_dict('records') if _fp.exists() else []
if _recovered:
    print(f'  Recovered {len(_recovered)} rows from {_fp.name}')

_rows = list(_recovered)
for mname, (factory, d_model, _) in MODEL_REGISTRY.items():
    if any(r['model'] == mname for r in _recovered):
        print(f'  Skipping {mname} (already done)')
        continue
    accs = []
    print(f'  [{mname}]')
    for seed in SEEDS:
        acc = run_mqar(factory, d_model, n, st, seed=seed,
                       log_every=max(300, st // 8))
        accs.append(round(acc, 4))
        print(f'    seed={seed}: {acc:.4f}')
    mean, std = float(np.mean(accs)), float(np.std(accs))
    print(f'    MEAN={mean:.4f} +/- {std:.4f}')
    _rows.append({'n_pairs': n, 'model': mname,
                  'mean': round(mean, 4), 'std': round(std, 4)})
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

pd.DataFrame(_rows).to_csv(_fp, index=False)
print(f'  Checkpoint: n{n}.csv')
if DEVICE == 'cuda':
    print(f'  Peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## 5c · Sweep: n=128

In [ ]:
# ── Sweep n=128 ───────────────────────────────────────────────────────────
n = 128
st = steps_for_n(n)
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
print()
print('=' * 60)
print(f' n={n}  steps={st:,}  T={3*n+1}  batch={batch_for_n(n)}  accum={BATCH//batch_for_n(n)}')
print('=' * 60)

_fp = OUT / 'logs' / f'n{n}.csv'
_recovered = pd.read_csv(_fp).to_dict('records') if _fp.exists() else []
if _recovered:
    print(f'  Recovered {len(_recovered)} rows from {_fp.name}')

_rows = list(_recovered)
for mname, (factory, d_model, _) in MODEL_REGISTRY.items():
    if any(r['model'] == mname for r in _recovered):
        print(f'  Skipping {mname} (already done)')
        continue
    accs = []
    print(f'  [{mname}]')
    for seed in SEEDS:
        acc = run_mqar(factory, d_model, n, st, seed=seed,
                       log_every=max(300, st // 8))
        accs.append(round(acc, 4))
        print(f'    seed={seed}: {acc:.4f}')
    mean, std = float(np.mean(accs)), float(np.std(accs))
    print(f'    MEAN={mean:.4f} +/- {std:.4f}')
    _rows.append({'n_pairs': n, 'model': mname,
                  'mean': round(mean, 4), 'std': round(std, 4)})
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

pd.DataFrame(_rows).to_csv(_fp, index=False)
print(f'  Checkpoint: n{n}.csv')
if DEVICE == 'cuda':
    print(f'  Peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## 5d · Sweep: n=200

In [ ]:
# ── Sweep n=200 ───────────────────────────────────────────────────────────
n = 200
st = steps_for_n(n)
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
print()
print('=' * 60)
print(f' n={n}  steps={st:,}  T={3*n+1}  batch={batch_for_n(n)}  accum={BATCH//batch_for_n(n)}')
print('=' * 60)

_fp = OUT / 'logs' / f'n{n}.csv'
_recovered = pd.read_csv(_fp).to_dict('records') if _fp.exists() else []
if _recovered:
    print(f'  Recovered {len(_recovered)} rows from {_fp.name}')

_rows = list(_recovered)
for mname, (factory, d_model, _) in MODEL_REGISTRY.items():
    if any(r['model'] == mname for r in _recovered):
        print(f'  Skipping {mname} (already done)')
        continue
    accs = []
    print(f'  [{mname}]')
    for seed in SEEDS:
        acc = run_mqar(factory, d_model, n, st, seed=seed,
                       log_every=max(300, st // 8))
        accs.append(round(acc, 4))
        print(f'    seed={seed}: {acc:.4f}')
    mean, std = float(np.mean(accs)), float(np.std(accs))
    print(f'    MEAN={mean:.4f} +/- {std:.4f}')
    _rows.append({'n_pairs': n, 'model': mname,
                  'mean': round(mean, 4), 'std': round(std, 4)})
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

pd.DataFrame(_rows).to_csv(_fp, index=False)
print(f'  Checkpoint: n{n}.csv')
if DEVICE == 'cuda':
    print(f'  Peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## 6 · Figure + Claim Verdict

In [ ]:
import glob

# ── Collect results from all sweep cells ─────────────────────────────────
rows = []
for fp in sorted(glob.glob(str(OUT / 'logs' / 'n*.csv'))):
    rows.extend(pd.read_csv(fp).to_dict('records'))
df = pd.DataFrame(rows)
print('Full results:')
print(df.to_string(index=False))

# ── Plot ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for mname, (_, d_model, col) in MODEL_REGISTRY.items():
    g = df[df.model == mname].sort_values('n_pairs')
    ax.errorbar(g.n_pairs, g['mean'], yerr=g['std'], color=col,
                marker='D', lw=2.5, ms=9, capsize=5, label=mname)

ax.axvline(UNIF_DH, color=C['uniform'], ls='--', lw=1.8, alpha=0.8,
           label=f'Uniform d_h={UNIF_DH}')
ax.axvline(SLOW_DH, color=C['hetero'], ls='--', lw=1.8, alpha=0.8,
           label=f'Hetero slow d_h={SLOW_DH}')
ax.axhline(0.80, color='gray', ls=':', lw=1.5)
ax.text(max(df.n_pairs) * 0.98, 0.82, 'pass (0.80)',
        ha='right', fontsize=9, color='gray')
ax.axhline(1 / (VOCAB - 1), color='lightgray', ls=':', lw=1)
ax.set(title='VLA Capacity: Hetero vs Uniform Heads (MQAR)',
       xlabel='n_pairs', ylabel='Accuracy (mean ± std, 3 seeds)')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9, loc='lower left')
plt.tight_layout()
plt.savefig(OUT / 'plots' / 'capacity_claim.pdf', bbox_inches='tight')
plt.savefig(OUT / 'plots' / 'capacity_claim.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: capacity_claim.{pdf,png}')

# ── Claim verdict ───────────────────────────────────────────────────
print()
print('CLAIM VERDICT:')
for cn in [128, 200]:
    gu = df[(df.model == 'Uniform-VLA') & (df.n_pairs == cn)]
    gh = df[(df.model == 'Hetero-VLA')  & (df.n_pairs == cn)]
    if not gu.empty and not gh.empty:
        u, h = gu['mean'].values[0], gh['mean'].values[0]
        verdict = 'PASS' if h > 0.80 else 'not yet >0.80'
        print(f'  n={cn}: Hetero={h:.3f}  Uniform={u:.3f}  '
              f'gap={h-u:+.3f}  {verdict}')

json.dump(df.to_dict('records'),
          open(OUT / 'manifest.json', 'w'), indent=2)
print(f'All files: {OUT}')